# Notebook 02: Feature Engineering & Stratified Dataset Splitting

**Project**: LinkSentinel (`LinkShield`)
**Objective**: Extract static numerical features from raw URLs and prepare reproducible 80% Train, 10% Validation, and 10% Test stratified splits (`random_state=42`).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

# Add src to path
sys.path.append('..')
from src.features.extract_features import URLLexicalFeatureExtractor

print("Feature engineering module loaded.")

## 1. Safety Boundary Proof

> **LinkSentinel Safety Principle**: Feature extraction relies 100% on static string parsing (`urllib.parse`, RegEx). Zero network HTTP requests, DNS lookups, or webpage downloads are performed.

In [ ]:
extractor = URLLexicalFeatureExtractor()
sample_url = "https://login.paypal.account-security.com/verify?token=123"
sample_feats = extractor.extract(sample_url)

print("Sample URL:", sample_url)
print("Extracted Features:")
for k, v in sample_feats.items():
    print(f"  {k}: {v}")

## 2. Process Experiment A (UCI Phishing Baseline Dataset)

Map target column `Result`: `-1` (Phishing/Suspicious) -> `1`, `1` (Legitimate/Safe-Looking) -> `0`.

In [ ]:
df_uci = pd.read_csv('../data/raw/uci_phishing_websites.csv')
X_uci = df_uci.drop(columns=['Result'])
y_uci = df_uci['Result'].map({-1: 1, 1: 0}).values  # 0 = Safe, 1 = Suspicious

# 80/10/10 Stratified Split
X_uci_train, X_uci_temp, y_uci_train, y_uci_temp = train_test_split(
    X_uci, y_uci, test_size=0.20, random_state=42, stratify=y_uci
)
X_uci_val, X_uci_test, y_uci_val, y_uci_test = train_test_split(
    X_uci_temp, y_uci_temp, test_size=0.50, random_state=42, stratify=y_uci_temp
)

print(f"Experiment A Splits -> Train: {X_uci_train.shape[0]}, Val: {X_uci_val.shape[0]}, Test: {X_uci_test.shape[0]}")

## 3. Process Experiment B (LinkSentinel Static URL Feature Extraction)

In [ ]:
df_raw_urls = pd.read_csv('../data/raw/raw_urls_dataset.csv')

features_list = []
for url in df_raw_urls['url']:
    feats = extractor.extract(url)
    features_list.append(feats)

X_urls = pd.DataFrame(features_list)
y_urls = df_raw_urls['target_label'].values

# 80/10/10 Stratified Split
X_url_train, X_url_temp, y_url_train, y_url_temp = train_test_split(
    X_urls, y_urls, test_size=0.20, random_state=42, stratify=y_urls
)
X_url_val, X_url_test, y_url_val, y_url_test = train_test_split(
    X_url_temp, y_url_temp, test_size=0.50, random_state=42, stratify=y_url_temp
)

print(f"Experiment B Splits -> Train: {X_url_train.shape[0]}, Val: {X_url_val.shape[0]}, Test: {X_url_test.shape[0]}")

## 4. Save Processed Splits to `data/processed/`

In [ ]:
splits_artifact = {
    'uci': {
        'X_train': X_uci_train, 'y_train': y_uci_train,
        'X_val': X_uci_val, 'y_val': y_uci_val,
        'X_test': X_uci_test, 'y_test': y_uci_test,
        'feature_names': list(X_uci.columns)
    },
    'url_engine': {
        'X_train': X_url_train, 'y_train': y_url_train,
        'X_val': X_url_val, 'y_val': y_url_val,
        'X_test': X_url_test, 'y_test': y_url_test,
        'feature_names': list(X_urls.columns)
    }
}

os.makedirs('../data/processed', exist_ok=True)
joblib.dump(splits_artifact, '../data/processed/splits.joblib')
print("Processed feature splits successfully persisted to data/processed/splits.joblib.")